In [ ]:
import copy
import csv
import os
import warnings
from argparse import ArgumentParser
import glob
import json
import random

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils import data
from tqdm import tqdm
import yaml

from nets import nn
from utils import util
from utils.valid import (
    compute_point_label_metrics_single,
    visualize_ground_truth_and_prediction_separately_detail_single,
    plot_training_progress,
)

# ── HIERARCHICAL 변경: 계층형 loss/checkpoint/data helper ─────────
from utils.hierarchical import (
    HierarchicalComputeLoss,
    load_flat_checkpoint_into_hierarchical,
)
from utils.ihc_dataset import (
    IHCHierarchicalDataset,
    discover_records,
    slide_id_from_stem,
    split_records_by_slide,
)

GPU_ID = 0  # HER2=0, ER/PR=1
if torch.cuda.is_available():
    if GPU_ID >= torch.cuda.device_count():
        raise RuntimeError(
            f"GPU_ID={GPU_ID}를 요청했지만 사용 가능한 GPU는 {torch.cuda.device_count()}개입니다."
        )
    torch.cuda.set_device(GPU_ID)
    device = torch.device(f"cuda:{GPU_ID}")
    print(f"device: {device} | {torch.cuda.get_device_name(GPU_ID)}")
else:
    device = torch.device("cpu")
    print("device: cpu")
with open('utils/detail_args.yaml', errors='ignore') as f:
    params = yaml.safe_load(f)

In [ ]:
input_size = 512

# ── 경로 설정 (IHC HER2) ────────────────────────────────────────
label_dir = '../../data/precise_BC_cell_scoring/her2/labels/'
image_dir = '../../data/precise_BC_cell_scoring/her2/patch_images/'
label_files = sorted(glob.glob(os.path.join(label_dir, '*.json')))

# ── 5 클래스 정의 (기존 노트북과 동일) ──────────────────────────
# Dataset label은 그대로 유지하고, model 내부에서 계층적으로 분리합니다.
class_names = {
    0: "class0",   # Tumor 0+
    1: "class1",   # Tumor 1+
    2: "class2",   # Tumor 2+
    3: "class3",   # Tumor 3+
    4: "other"     # Non-tumor
}
num_classes = len(class_names)
params['names'] = class_names

# ── HIERARCHICAL 변경: image/label은 lazy loading ────────────────
# 기존 노트북은 전체 이미지를 RAM에 올렸지만, 동일 데이터 구조를 유지하면서
# 각 batch에서 이미지를 읽어 메모리 사용량을 줄입니다.
records = discover_records(image_dir, label_dir)
image_filenames = [str(image_path) for image_path, _ in records]

print(f"📂 {len(records)} image/label pairs found")
print(f"   image: {image_filenames[0]}")
print(f"   label: {records[0][1]}")

# was_nonT=True가 실제로 other로 읽히는지 확인
first_labels = IHCHierarchicalDataset.load_labels(records[0][1])
print(f"✅ first label shape: {first_labels.shape}")

In [ ]:
# 기존 collate_fn1 이름을 유지합니다.
def collate_fn1(batch):
    return IHCHierarchicalDataset.collate_fn(batch)


# ── Train / Val split ────────────────────────────────────────────
# HIERARCHICAL 변경: 인접 patch leakage를 막기 위해 slide 단위로 분리
train_records, val_records = split_records_by_slide(
    records, val_fraction=0.1, seed=242
)
train_slides = {slide_id_from_stem(record[0].stem) for record in train_records}
val_slides = {slide_id_from_stem(record[0].stem) for record in val_records}
assert train_slides.isdisjoint(val_slides)

train_dataset = IHCHierarchicalDataset(
    train_records, input_size=input_size, augment=True
)
val_dataset = IHCHierarchicalDataset(
    val_records, input_size=input_size, augment=False
)

# ── 클래스 분포 분석 (기존 노트북과 동일한 위치) ────────────────
print("\n" + "=" * 70)
print("📊 클래스 분포 분석")
print("=" * 70)

total_counts = np.zeros(num_classes, dtype=np.float64)
for _, label_path in tqdm(train_records, desc='Counting train labels'):
    label_array = IHCHierarchicalDataset.load_labels(label_path)
    if len(label_array):
        total_counts += np.bincount(
            label_array[:, 0].astype(int), minlength=num_classes
        )

total_n = total_counts.sum()
for i in range(num_classes):
    print(f"  [{i}] {class_names[i]:20s}: {int(total_counts[i]):8d}개  "
          f"({total_counts[i] / total_n * 100:.2f}%)")

print("\n계층형 label 해석:")
print(f"  Tumor     (class0~3): {int(total_counts[:4].sum()):,}")
print(f"  Non-tumor (other)   : {int(total_counts[4]):,}")
print("  ※ WeightedRandomSampler + class weight 이중 보정은 사용하지 않습니다.")
print("  ※ Non-tumor → Tumor 비용은 FALSE_TUMOR_WEIGHT로 직접 제어합니다.")
print("=" * 70)

# ── DataLoader ───────────────────────────────────────────────────
batch_size = 16
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=4, collate_fn=collate_fn1, drop_last=True,
    pin_memory=(device.type == 'cuda'), persistent_workers=True,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=4, collate_fn=collate_fn1, drop_last=False,
    pin_memory=(device.type == 'cuda'), persistent_workers=True,
)

print(f"\n📦 DataLoaders ready:")
print(f"  Train patches : {len(train_dataset):,} ({len(train_slides)} slides)")
print(f"  Val patches   : {len(val_dataset):,} ({len(val_slides)} slides)")
print(f"  Train batches : {len(train_loader):,}")
print(f"  Val batches   : {len(val_loader):,}")

In [ ]:
def visualize_sample_with_overlay(dataset, index=0):
    image_tensor, cls_tensor, box_tensor, _ = dataset[index]

    image = image_tensor.numpy().transpose(1, 2, 0)
    cls = cls_tensor.numpy()
    boxes = box_tensor.numpy()
    height, width = image.shape[:2]

    colors = ['red', 'limegreen', 'yellow', 'magenta', 'dodgerblue']

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(image.astype(np.uint8))

    for i in range(len(boxes)):
        cx_norm, cy_norm, w_norm, h_norm = boxes[i]
        cx_px = cx_norm * width
        cy_px = cy_norm * height
        w_px = w_norm * width
        h_px = h_norm * height
        x1 = cx_px - w_px / 2
        y1 = cy_px - h_px / 2

        class_id = int(cls[i])
        rect = patches.Rectangle(
            (x1, y1), w_px, h_px, linewidth=2,
            edgecolor=colors[class_id], facecolor='none'
        )
        ax.add_patch(rect)

    ax.set_title(f'Sample {index} | Total Objects: {len(boxes)}', fontsize=12)
    ax.axis('off')

    legend_elements = [
        patches.Patch(
            color=colors[i],
            label=f'{class_names[i]} ({int((cls == i).sum())})'
        ) for i in range(num_classes)
    ]
    fig.legend(
        handles=legend_elements, loc='lower center', ncol=3,
        bbox_to_anchor=(0.5, -0.05), fontsize=9
    )
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    plt.show()

    print(f"\n📊 Sample {index}:")
    print(f"  이미지 크기 : {width} x {height}")
    print(f"  총 객체 수  : {len(boxes)}")
    for i in range(num_classes):
        count = int((cls == i).sum())
        if count > 0:
            print(f"  [{i}] {class_names[i]:20s}: {count}개")


visualize_sample_with_overlay(train_dataset, index=0)

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── HIERARCHICAL 변경: loss 구성 ─────────────────────────────────
# objectness : cell / background
# tumor      : Tumor / Non-tumor
# grade      : Tumor cell에만 0+/1+/2+/3+
# false_tumor_weight: Non-tumor를 Tumor로 보내는 오류 비용
FALSE_TUMOR_WEIGHT = 1.25
LEARNING_RATE = 1e-4
MIN_LEARNING_RATE = 1e-6
MAX_EPOCHS = 2000
EARLY_STOPPING_PATIENCE = 20
VALIDATION_TUMOR_THRESHOLD = 0.50
params['objectness'] = 1.0
params['tumor'] = 1.0
params['grade'] = 1.0
params['false_tumor_weight'] = FALSE_TUMOR_WEIGHT
params['top_k'] = 10
params['assigner_alpha'] = 0.5
params['assigner_beta'] = 6.0

# ── HIERARCHICAL 변경: 일반 yolo_v11_m(5) 대신 계층형 model ─────
model = nn.yolo_v11_m_hierarchical(num_grades=4).to(device)
print(f"PID={os.getpid()} | declared={device} | model={next(model.parameters()).device}")

# 기존 flat 5-class checkpoint에서 backbone/FPN/head feature 이관
flat_checkpoint_path = '../../model/precise_BC_cell_scoring/her2_yolov11/best_model.pt'
if os.path.exists(flat_checkpoint_path):
    transfer_report = load_flat_checkpoint_into_hierarchical(
        model, flat_checkpoint_path, map_location=device
    )
    print(f"✅ 기존 checkpoint 이관: {transfer_report}")
else:
    print("⚠️ 기존 flat checkpoint가 없어 random initialization으로 시작합니다.")

optimizer = torch.optim.AdamW(
    util.set_params(model, params['weight_decay']),
    lr=LEARNING_RATE, betas=(0.9, 0.999),
    weight_decay=params['weight_decay']
)
criterion = HierarchicalComputeLoss(model, params)

cosine_scheduler = CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=MIN_LEARNING_RATE
)

# 기존 노트북과 동일한 one-batch 확인
model.train()
smoke_images, smoke_targets = next(iter(train_loader))
smoke_images = smoke_images.to(device).float() / 255.0
with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
    smoke_outputs = model(smoke_images)
    smoke_losses = criterion(smoke_outputs, smoke_targets)
print("\n✅ Hierarchical loss smoke test:")
for loss_name, loss_value in smoke_losses.items():
    print(f"  {loss_name:12s}: {loss_value.item():.4f}")
del smoke_images, smoke_targets, smoke_outputs, smoke_losses
if device.type == 'cuda':
    torch.cuda.empty_cache()

print(f"\n🎯 Hierarchical 학습 설정:")
print(f"  ✅ Objectness loss : {params['objectness']}")
print(f"  ✅ Tumor loss      : {params['tumor']}")
print(f"  ✅ Grade loss      : {params['grade']}")
print(f"  ✅ False-T weight  : {params['false_tumor_weight']}")
print(f"  ✅ Learning rate   : {LEARNING_RATE} → {MIN_LEARNING_RATE}")
print(f"  ✅ Epochs/Patience : {MAX_EPOCHS} / {EARLY_STOPPING_PATIENCE}")
print(f"  ✅ Val tumor gate  : {VALIDATION_TUMOR_THRESHOLD}")
print(f"  ✅ Assigner        : top_k={params['top_k']}, alpha={params['assigner_alpha']}")

In [ ]:
train_losses = []
train_box_losses = []
train_objectness_losses = []
train_tumor_losses = []
train_grade_losses = []
train_dfl_losses = []
val_det_recalls = []
val_cls_accs = []
val_macro_f1s = []
val_macro_precisions = []
val_macro_recalls = []
val_other_recalls = []
val_tumor_recalls = []
val_gate_scores = []
val_class_stats_history = []

epochs = MAX_EPOCHS
save_dir = '../../model/precise_BC_cell_scoring/her2_hierarchical_yolov11/'
os.makedirs(save_dir, exist_ok=True)

start_epoch = 0
best_hierarchical_score = 0.0
epochs_without_improvement = 0

# ── 체크포인트 불러오기 (재개 시 주석 해제) ─────────────────────
# checkpoint_path = os.path.join(save_dir, 'last_model.pt')
# if os.path.exists(checkpoint_path):
#     checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
#     model.load_state_dict(checkpoint['model_state_dict'])
#     optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#     cosine_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
#     start_epoch = checkpoint['epoch'] + 1
#     best_hierarchical_score = checkpoint.get('best_hierarchical_score', 0.0)
#     epochs_without_improvement = checkpoint.get('epochs_without_improvement', 0)

accumulate = max(round(64 / batch_size), 1)
amp_scale = torch.amp.GradScaler(enabled=(device.type == 'cuda'))
print(f"Gradient accumulation steps: {accumulate}")

for epoch in range(start_epoch, epochs):
    model.train()

    avg_box_loss = util.AverageMeter()
    avg_objectness_loss = util.AverageMeter()
    avg_tumor_loss = util.AverageMeter()
    avg_grade_loss = util.AverageMeter()
    avg_dfl_loss = util.AverageMeter()
    avg_total_loss = util.AverageMeter()

    train_pbar = tqdm(
        enumerate(train_loader), total=len(train_loader),
        desc=f'Epoch {epoch + 1}/{epochs} Training'
    )
    optimizer.zero_grad(set_to_none=True)

    for i, (torch_images, targets) in train_pbar:
        torch_images = torch_images.to(device, non_blocking=True).float() / 255.0

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(torch_images)
            losses = criterion(outputs, targets)
            total_loss = losses['total'] / accumulate

        avg_box_loss.update(losses['box'].item(), torch_images.size(0))
        avg_objectness_loss.update(losses['objectness'].item(), torch_images.size(0))
        avg_tumor_loss.update(losses['tumor'].item(), torch_images.size(0))
        avg_grade_loss.update(losses['grade'].item(), torch_images.size(0))
        avg_dfl_loss.update(losses['dfl'].item(), torch_images.size(0))
        avg_total_loss.update(losses['total'].item(), torch_images.size(0))

        amp_scale.scale(total_loss).backward()
        if (i + 1) % accumulate == 0 or (i + 1) == len(train_loader):
            amp_scale.step(optimizer)
            amp_scale.update()
            optimizer.zero_grad(set_to_none=True)

        memory = (
            f'{torch.cuda.memory_reserved() / 1E9:.4g}G'
            if device.type == 'cuda' else 'CPU'
        )
        status = (
            f'Memory: {memory} | Box: {avg_box_loss.avg:.3f} | '
            f'Obj: {avg_objectness_loss.avg:.3f} | '
            f'Tumor: {avg_tumor_loss.avg:.3f} | '
            f'Grade: {avg_grade_loss.avg:.3f} | DFL: {avg_dfl_loss.avg:.3f}'
        )
        train_pbar.set_description(f'Epoch {epoch + 1}/{epochs} | {status}')

    cosine_scheduler.step()

    train_losses.append(avg_total_loss.avg)
    train_box_losses.append(avg_box_loss.avg)
    train_objectness_losses.append(avg_objectness_loss.avg)
    train_tumor_losses.append(avg_tumor_loss.avg)
    train_grade_losses.append(avg_grade_loss.avg)
    train_dfl_losses.append(avg_dfl_loss.avg)

    # ── Validation (test와 동일한 explicit Tumor gate 적용) ──────
    try:
        point_metrics = compute_point_label_metrics_single(
            model, val_loader, device, params, distance_threshold=16,
            class_agnostic_nms=True,
            tumor_threshold=VALIDATION_TUMOR_THRESHOLD
        )
        detection_recall = point_metrics.get('detection_recall', 0)
        cls_accuracy = point_metrics.get('classification_accuracy', 0)
        macro_precision = point_metrics.get('macro_precision', 0)
        macro_recall = point_metrics.get('macro_recall', 0)
        macro_f1 = point_metrics.get('macro_f1', 0)
        overall_recall = point_metrics.get('overall_recall', 0)
        class_stats = point_metrics.get('class_stats', {})
        tumor_recall = point_metrics.get('gate_tumor_recall', 0)
        other_recall = point_metrics.get('gate_other_recall', 0)
        gate_score = point_metrics.get('gate_score', 0)

        val_det_recalls.append(detection_recall)
        val_cls_accs.append(cls_accuracy)
        val_macro_f1s.append(macro_f1)
        val_macro_precisions.append(macro_precision)
        val_macro_recalls.append(macro_recall)
        val_other_recalls.append(other_recall)
        val_tumor_recalls.append(tumor_recall)
        val_gate_scores.append(gate_score)
        val_class_stats_history.append(class_stats)

        # 양방향 gate, grade 분류, detection을 함께 보는 균형 점수
        hierarchical_score = (
            0.5 * gate_score + 0.3 * macro_f1 + 0.2 * detection_recall
        )

        print(f"\nEpoch {epoch + 1}/{epochs}:")
        print(
            f"  Train Loss — Box: {avg_box_loss.avg:.4f} | "
            f"Obj: {avg_objectness_loss.avg:.4f} | "
            f"Tumor: {avg_tumor_loss.avg:.4f} | "
            f"Grade: {avg_grade_loss.avg:.4f} | DFL: {avg_dfl_loss.avg:.4f}"
        )
        print(f"  Detection Recall       : {detection_recall:.4f}")
        print(f"  Classification Accuracy: {cls_accuracy:.4f}")
        print(f"  Tumor Gate Recall      : {tumor_recall:.4f}")
        print(f"  Other Gate Recall      : {other_recall:.4f}")
        print(f"  Gate Balanced Score    : {gate_score:.4f} ⭐")
        print(f"  Macro Precision        : {macro_precision:.4f}")
        print(f"  Macro Recall           : {macro_recall:.4f}")
        print(f"  Macro F1               : {macro_f1:.4f}")
        print(f"  Hierarchical Score     : {hierarchical_score:.4f}")
    except Exception as e:
        print(f"Validation 오류: {e}")
        detection_recall = cls_accuracy = 0
        macro_precision = macro_recall = macro_f1 = overall_recall = 0
        tumor_recall = other_recall = gate_score = hierarchical_score = 0
        class_stats = {}

    # ── Checkpoint ──────────────────────────────────────────────
    improved = hierarchical_score > best_hierarchical_score
    if improved:
        best_hierarchical_score = hierarchical_score
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    ckpt_data = {
        'architecture': 'hierarchical_cell_tumor_grade',
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': cosine_scheduler.state_dict(),
        'amp_scale_state_dict': amp_scale.state_dict(),
        'train_loss': avg_total_loss.avg,
        'box_loss': avg_box_loss.avg,
        'objectness_loss': avg_objectness_loss.avg,
        'tumor_loss': avg_tumor_loss.avg,
        'grade_loss': avg_grade_loss.avg,
        'dfl_loss': avg_dfl_loss.avg,
        'detection_recall': detection_recall,
        'classification_accuracy': cls_accuracy,
        'tumor_recall': tumor_recall,
        'other_recall': other_recall,
        'gate_score': gate_score,
        'tumor_threshold': VALIDATION_TUMOR_THRESHOLD,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'overall_recall': overall_recall,
        'hierarchical_score': hierarchical_score,
        'best_hierarchical_score': best_hierarchical_score,
        'epochs_without_improvement': epochs_without_improvement,
        'class_stats': class_stats,
        'params': params,
    }
    torch.save(ckpt_data, os.path.join(save_dir, 'last_model.pt'))

    if improved:
        torch.save(ckpt_data, os.path.join(save_dir, 'best_model.pt'))
        print(f"🎉 새로운 베스트! Hierarchical Score: {hierarchical_score:.4f}")
    else:
        print(
            f"  Early stopping: {epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE} epochs without improvement"
        )

    # ── 시각화 (기존 노트북과 동일하게 10 epoch마다) ─────────────
    if (epoch + 1) % 10 == 0:
        try:
            sample_idx = random.randint(0, len(val_dataset) - 1)
            visualize_ground_truth_and_prediction_separately_detail_single(
                model, val_dataset, idx=sample_idx,
                epoch=epoch + 1, save_dir=save_dir
            )
        except Exception as e:
            print(f"시각화 오류: {e}")

    # ── 학습 곡선 (10 epoch마다) ────────────────────────────────
    if (epoch + 1) % 10 == 0:
        try:
            plot_training_progress(
                train_losses, val_det_recalls, val_cls_accs,
                val_macro_precisions, val_macro_recalls, val_macro_f1s,
                epoch + 1, save_dir,
                class_stats_history=val_class_stats_history
            )
        except Exception as e:
            print(f"그래프 오류: {e}")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"⏹️ Early stopping at epoch {epoch + 1}: "
            f"{EARLY_STOPPING_PATIENCE} epochs 동안 score 개선 없음"
        )
        break

print("🎯 계층형 학습 완료!")
print(f"  Best Hierarchical Score : {best_hierarchical_score:.4f}")
print(f"  저장 경로              : {save_dir}")